In [1]:
import pangolin
from pangolin import interface as pi
import numpyro
from numpyro import distributions as ndist
import jax
from jax import numpy as jnp
import numpy as np
import functools

## Simple, minimal, math-like API

Suppose that you've got this model:

$ m \sim \text{normal}(0,1)$, 

$ s \sim \text{lognormal}(0,1)$, 

$ x \sim \text{normal}(m,s)$.

Then what is $\mathbb{E}[m \vert x=3.3]$? Easy:

In [2]:
m = pi.normal(0,1)
s = pi.lognormal(0,1)
x = pi.normal(m,s)
pangolin.blackjax.E(m, x, 3.3)

Array(0.75575864, dtype=float32)

## Radically simple, inspectable IR

The core idea behind pangolin is a radically simple intermediate representation (IR) for probabilistic model. Fundamentally, there are only two abstractions.

1. An `Op` denotes some pure function or conditional probability distribution.
2. An `RV` is an `Op` plus some tuple of parent `RV`s.

There's a graph of `RV`. That is it. There is no other abstraction for a "model". 

The IR is immutable. After creating random variables you can look at them.

In [3]:
x = pi.normal(0,1)
print(f"{x.op=}")
print(f"{x.shape=}")

x.op=Normal()
x.shape=()


In [4]:
x = pi.multi_normal(np.zeros(3), np.eye(3))
print(f"{x.op=}")
print(f"{x.shape=}")
print(f"{x.parents[0]=}")
print(f"{x.parents[1]=}")

x.op=MultiNormal()
x.shape=(3,)
x.parents[0]=InfixRV(Constant([0.,0.,0.]))
x.parents[1]=InfixRV(Constant([[1.,0.,0.],[0.,1.,0.],[0.,0.,1.]]))


In [5]:
x = pi.vfor(lambda i, j: pi.normal(i, pi.exp(j)), i=3, j=5)
pangolin.ir.graph_upstream(x)

○   ()     abstract_op
│
│ ○ ()     abstract_op
│ │
│ ○ ()     exp
│ │
└─○ (3, 5) vmap(vmap(normal, [None, None], 5), [None, None], 3)


In [6]:
x = pi.vfor(lambda i: pi.exp(i), i=3)
x

InfixRV(VMap(Exp(), [None], 3), InfixRV(AbstractOp()))

In [7]:
x = pi.vfor(lambda i, j: pi.normal(i, i), i=3, j=4)
x

InfixRV(VMap(VMap(Normal(), [None, None], 4), [None, None], 3), InfixRV(AbstractOp()), InfixRV(AbstractOp()))

In [8]:
x = pi.vfor(lambda i, j: pi.exp(i), i=3, j=4)
x


InfixRV(VMap(VMap(Exp(), [None], 4), [None], 3), InfixRV(AbstractOp()))

In [9]:
x = pi.vfor(lambda i, j: pi.exp(j), i=3, j=4)
x


InfixRV(VMap(VMap(Exp(), [None], 4), [None], 3), InfixRV(AbstractOp()))

In [10]:
a = pi.constant([1,2,3])
x = pi.vfor(lambda i, j: pi.exp(a[i]), i=3, j=4)
x

InfixRV(VMap(VMap(Exp(), [None], 4), [0], 3), InfixRV(Constant([1,2,3])))

In [11]:
x = pi.vfor(lambda i, j: pi.exp(i), i=3, j=4)
x

InfixRV(VMap(VMap(Exp(), [None], 4), [None], 3), InfixRV(AbstractOp()))

In [12]:
pi.vfor(lambda i,j: j, i=3, j=4)

InfixRV(VMap(VMap(AbstractOp(), [], 4), [], 3))

In [13]:
pi.vfor(lambda i,j: i, i=3, j=4)

InfixRV(VMap(VMap(AbstractOp(), [], 4), [], 3))

In [14]:
pi.vfor(lambda i: i, i=3)

InfixRV(VMap(AbstractOp(), [], 3))

In [15]:
pi.vfor(lambda i: pi.exp(i), i=3)

InfixRV(VMap(Exp(), [None], 3), InfixRV(AbstractOp()))

In [16]:
a = pi.constant([0,1,2])
pi.vfor(lambda i: pi.exp(a[i]), i=3)

InfixRV(VMap(Exp(), [0], 3), InfixRV(Constant([0,1,2])))

In [17]:
import jax
from jax import numpy as jnp
jax.vmap(lambda i: 2.0)(jnp.array([0,1,2]))

Array([2., 2., 2.], dtype=float32, weak_type=True)

In [18]:
pi.vmap(lambda i: pi.constant(2.0))(pi.constant([0,1,2]))

InfixRV(VMap(Constant(2.), [], 3))